# Assignment 2 - Problem 1

## Part a

Let the neural network output be
$$
g(x)=(g_1(x),g_2(x))\in\mathbb R^2.
$$

We define the following link functions to convert the network outputs into the conditional mean and standard deviation:

1. Conditional mean: Use the identity function, since the mean can be any real number.
$$
\boxed{f_\mu(x)=g_1(x)}
$$

2. Conditional standard deviation: Use the exponential function to ensure the standard deviation is strictly positive.
$$
\boxed{f_\sigma(x)=\exp(g_2(x))}
$$

So, the appropriate link functions are the identity function for $f_\mu(x)$ and the exponential function for $f_\sigma(x)$. This ensures $f_\mu(x)\in\mathbb R$ and $f_\sigma(x)>0$.

## Part b

Given the assumption:
$$
Y\mid X=x\sim N(f_\mu(x),f_\sigma(x)^2)
$$

We use the Gaussian Negative Log-Likelihood as the loss function.

The conditional probability density is:
$$
p(y\mid x)=\frac{1}{\sqrt{2\pi}f_\sigma(x)}
\exp\left(-\frac{(y-f_\mu(x))^2}{2f_\sigma(x)^2}\right)
$$

Taking the negative logarithm gives:
$$
\ell(x,y)=\frac12\log(2\pi)+\log f_\sigma(x)+\frac{(y-f_\mu(x))^2}{2f_\sigma(x)^2}
$$

So, for a training dataset of $n$ observations, the loss function is
$$
\boxed{
L=\frac1n\sum_{i=1}^{n}
\left[
\log f_\sigma(x_i)+
\frac{(y_i-f_\mu(x_i))^2}{2f_\sigma(x_i)^2}
\right]
}
$$

where the constant $\frac12\log(2\pi)$ is omitted because it does not affect optimization.

This loss trains the model to predict both the conditional mean and the conditional standard deviation.

## Part c

In [ ]:
import pandas as pd

# Read the parquet dataset
df = pd.read_parquet('/content/taxi_trips.parquet')

df.head()

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration,min_of_day,day_of_week,day_of_year
187173,0.270744,1.425051,0.466831,0.776192,7.965142,-0.942057,-1.047619,1.180855
179655,0.085681,0.306079,-0.226274,0.124111,10.680277,0.778703,-1.047619,-0.311290
120048,-0.555533,-0.934806,-0.022489,-0.548561,5.716184,-1.183016,-1.047619,1.316504
62587,0.069718,0.035804,0.258900,0.317284,4.266709,-0.908008,0.481657,0.018144
212063,-0.234482,0.074616,0.115836,0.258310,11.629685,1.066806,-0.537860,0.521985


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# 1. Prepare data splits
target_col = 'trip_duration'
feature_cols = [c for c in df.columns if c != target_col]

X = df[feature_cols].values
y = df[target_col].values

# Train (70%) / Val (15%) / Test (15%)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1765, random_state=42) # 0.15 / 0.85 approx 17.65%

print(f"Train size: {X_train.shape[0]}, Val size: {X_val.shape[0]}, Test size: {X_test.shape[0]}")

# Convert to PyTorch tensors
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

# 2. Baseline Model (Predicts global mean and global std of training target)
baseline_mu = np.mean(y_train)
baseline_sigma = np.std(y_train)

def gaussian_nll(y_true, mu, sigma):
    # Constant 0.5 * log(2 * pi) omitted
    return np.mean(np.log(sigma) + ((y_true - mu) ** 2) / (2 * (sigma ** 2)))

baseline_train_nll = gaussian_nll(y_train, baseline_mu, baseline_sigma)
baseline_val_nll = gaussian_nll(y_val, baseline_mu, baseline_sigma)
baseline_test_nll = gaussian_nll(y_test, baseline_mu, baseline_sigma)

print(f"\nBaseline Model (Predicting global Train Statistics):\n"
      f"Predicted Mean: {baseline_mu:.4f}, Predicted Std: {baseline_sigma:.4f}\n"
      f"Train NLL: {baseline_train_nll:.4f} | Val NLL: {baseline_val_nll:.4f} | Test NLL: {baseline_test_nll:.4f}\n")

# 3. Define the MLP Network
class TaxiTripMLP(nn.Module):
    def __init__(self, input_dim):
        super(TaxiTripMLP, self).__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        # Branch 1: Output mean (no activation constraint)
        self.mu_layer = nn.Linear(32, 1)
        # Branch 2: Output log variance or log std (to guarantee strict positivity via exp)
        self.log_std_layer = nn.Linear(32, 1)

    def forward(self, x):
        feat = self.shared(x)
        mu = self.mu_layer(feat).squeeze(-1)
        # We output log_std to stabilize training. exp(log_std) will act as f_sigma(x)
        log_std = self.log_std_layer(feat).squeeze(-1)
        # Limit log_std values to prevent overflow/underflow
        log_std = torch.clamp(log_std, min=-10.0, max=10.0)
        sigma = torch.exp(log_std)
        return mu, sigma

# 4. Custom Gaussian NLL Loss Function
def custom_gaussian_nll_loss(mu, sigma, target):
    # loss = log(sigma) + (y - mu)^2 / (2 * sigma^2)
    variance = sigma ** 2
    loss = torch.log(sigma) + ((target - mu) ** 2) / (2 * variance)
    return torch.mean(loss)

# 5. Initialize Model and Optimizer
model = TaxiTripMLP(input_dim=X_train.shape[1]).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 6. Training Loop
epochs = 10
for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        mu, sigma = model(batch_X)
        loss = custom_gaussian_nll_loss(mu, sigma, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation step
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            mu, sigma = model(batch_X)
            loss = custom_gaussian_nll_loss(mu, sigma, batch_y)
            val_loss += loss.item() * batch_X.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# 7. Evaluate on the Test Set
model.eval()
test_loss = 0.0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        mu, sigma = model(batch_X)
        loss = custom_gaussian_nll_loss(mu, sigma, batch_y)
        test_loss += loss.item() * batch_X.size(0)
test_loss /= len(test_loader.dataset)

print(f"\nTest Set Results:\nMLP model Test NLL Loss: {test_loss:.4f} | Baseline Test NLL Loss: {baseline_test_nll:.4f}")

Train size: 366473, Val size: 78547, Test size: 78534

Baseline Model (Predicting global Train Statistics):
Predicted Mean: 10.4106, Predicted Std: 7.7517
Train NLL: 2.5479 | Val NLL: 2.5453 | Test NLL: 2.5526

Epoch 01/10 | Train Loss: 3.0616 | Val Loss: 2.0313
Epoch 02/10 | Train Loss: 1.9110 | Val Loss: 1.8413
Epoch 03/10 | Train Loss: 1.8259 | Val Loss: 1.7963
Epoch 04/10 | Train Loss: 1.7943 | Val Loss: 1.7618
Epoch 05/10 | Train Loss: 1.7794 | Val Loss: 1.7540
Epoch 06/10 | Train Loss: 1.7616 | Val Loss: 1.7442
Epoch 07/10 | Train Loss: 1.7413 | Val Loss: 1.7349
Epoch 08/10 | Train Loss: 1.7354 | Val Loss: 1.7202
Epoch 09/10 | Train Loss: 1.7195 | Val Loss: 1.7393
Epoch 10/10 | Train Loss: 1.7111 | Val Loss: 1.7211

Test Set Results:
MLP model Test NLL Loss: 1.8128 | Baseline Test NLL Loss: 2.5526


## Part d

To find the probability that a trip will take less than 45 minutes under our model's assumption, we can use the Cumulative Distribution Function (CDF) of the normal distribution.

Since our model assumes:
$$Y \mid X = x \sim N\big(f_\mu(x), f_\sigma(x)^2\big)$$

The probability is given by:
$$P(Y < 45 \mid X = x) = \Phi\left(\frac{45 - f_\mu(x)}{f_\sigma(x)}\right)$$

where $\Phi$ is the Cumulative Distribution Function of the standard normal distribution $N(0, 1)$.

## Part e

In [ ]:
from sklearn.metrics import brier_score_loss, accuracy_score

# Put the model in evaluation mode
model.eval()

all_predicted_probs = []
all_true_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)

        # Get model outputs (mean and standard deviation)
        mu, sigma = model(batch_X)

        # Create Normal distribution objects
        dist = torch.distributions.Normal(mu, sigma)

        # Compute the probability P(Y < 45)
        prob_less_than_45 = dist.cdf(torch.tensor(45.0, device=device))

        all_predicted_probs.extend(prob_less_than_45.cpu().numpy())
        # True label: 1 if trip duration is less than 45, 0 otherwise
        all_true_labels.extend((batch_y < 45.0).long().cpu().numpy())

all_predicted_probs = np.array(all_predicted_probs)
all_true_labels = np.array(all_true_labels)

# Calculate evaluation metrics
brier = brier_score_loss(all_true_labels, all_predicted_probs)
predictions_binary = (all_predicted_probs >= 0.5).astype(int)
accuracy = accuracy_score(all_true_labels, predictions_binary)

print(f"--- Held-out Test Set Evaluation (Target: Trip Duration < 45 mins) ---")
print(f"Brier Score: {brier:.6f}")
print(f"Classification Accuracy: {accuracy * 100:.4f}%")

# Also let's see how many trips in our test set actually took less than 45 mins
base_rate = np.mean(all_true_labels)
print(f"Base Rate (proportion of test trips < 45 mins): {base_rate * 100:.4f}%")

--- Held-out Test Set Evaluation (Target: Trip Duration < 45 mins) ---
Brier Score: 0.003218
Classification Accuracy: 99.6142%
Base Rate (proportion of test trips < 45 mins): 99.5760%
